##### Init



In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS dbr_dev.yanquiel")
spark.sql(f"CREATE VOLUME IF NOT EXISTS dbr_dev.yanquiel.raw_data")

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import StructType, StringType, IntegerType, DoubleType
import requests
import pandas as pd

catalog = "dbr_dev"
schema = "yanquiel"
volume_path = f"/Volumes/{catalog}/{schema}/raw_data/world_population.csv"

##### Bronze layer


In [0]:
population_schema = (
    StructType()
    .add("Rank", IntegerType())
    .add("CCA3", StringType())
    .add("Country_Territory", StringType())
    .add("Capital", StringType())
    .add("Continent", StringType())
    .add("Population_2022", IntegerType())
    .add("Population_2020", IntegerType())
    .add("Population_2015", IntegerType())
    .add("Population_2010", IntegerType())
    .add("Population_2000", IntegerType())
    .add("Population_1990", IntegerType())
    .add("Population_1980", IntegerType())
    .add("Population_1970", IntegerType())
    .add("Area_km2", IntegerType())
    .add("Density_per_km2", DoubleType())
    .add("Growth_Rate", DoubleType())
    .add("World_Population_Percentage", DoubleType())
)

df_bronze = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(population_schema)
    .load(volume_path)
)

df_bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.brz_world_population")

response = requests.get("https://countries.dev/countries", timeout=15)
response.raise_for_status()
data = response.json()

df_bronze_api = spark.createDataFrame(pd.DataFrame(data))
df_bronze_api.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.brz_countries_api")


print(f"Bronze - population: {df_bronze.count()} rows, API: {df_bronze_api.count()} rows")

##### Silver layer


In [0]:
RENAME_MAP = {
    "Country_Territory": "country",
    "Population_2022": "population_2022",
    "Area_km2": "area_km2",
    "Density_per_km2": "density_km2",
    "Growth_Rate": "growth_rate",
    "World_Population_Percentage": "world_population_pct"
}

df_population_silver = spark.table(f"{catalog}.{schema}.brz_world_population")
for old_name, new_name in RENAME_MAP.items():
    df_population_silver = df_population_silver.withColumnRenamed(old_name, new_name)

for field in df_population_silver.schema.fields:
    if isinstance(field.dataType, T.StringType):
        df_population_silver = df_population_silver.withColumn(field.name, F.trim(F.col(field.name)))

df_population_silver = (
    df_population_silver.select(
        "CCA3", "country", "Capital", "Continent",
        "population_2022", "area_km2", "density_km2",
        "growth_rate", "world_population_pct"
    )
    .dropDuplicates()
    .dropna(subset=["country", "CCA3"])
)

df_api = (
    spark.table(f"{catalog}.{schema}.brz_countries_api")
    .select(
        "alpha3Code", "capital", "region",
        F.col("currencies")[0]["code"].alias("currency_code")
    )
    .dropDuplicates()
    .dropna(subset=["alpha3Code"])
)

df_silver = (
    df_population_silver .alias("p")
    .join(df_api.alias("a"), F.col("p.CCA3") == F.col("a.alpha3Code"), "left")
    .select(
        "p.CCA3", "p.country", "p.Continent",
        "p.population_2022", "p.area_km2", "p.density_km2", "p.growth_rate",
        "a.capital", "a.currency_code", "a.region"
    )
)

df_silver.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.slv_population")

df_silver.printSchema()

In [0]:
df_check_slv = spark.table(f"{catalog}.{schema}.slv_population")
df_check_slv.limit(5).display()

##### Gold layer

In [0]:
df_silver = spark.table(f"{catalog}.{schema}.slv_population")


df = (
    df_silver.groupBy("Continent")
    .agg(
        F.sum("population_2022").alias("continent_total_population"),
        F.count("country").alias("continent_num_countries")
    )
)


df_gold = df_silver.join(df, on="Continent", how="left")

df_gold.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gld_population")
df_gold.printSchema()


In [0]:
df_check_gld = spark.table(f"{catalog}.{schema}.gld_population")
df_check_gld.limit(5).display()